# 01_A TOPIS 원본 확인

`crawling/topis_speed.py`로 받은 45개월 CSV를 확인한다.

1. 수집 목록 요약
2. `~08시` 열이 몇 시의 평균인지 (월간 보고서와 대조)
3. 서울 전체가 동시에 비는 시간 찾기 -> `config/topis_gaps.csv`

CSV는 릴리스 `topis-data-v1`에서 받아 `data/raw/topis/csv/`에 풀어 두면 된다.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.io_utils import TOPIS_CSV_DIR, HOUR_COLS, read_topis

# read_topis: 2024-02 이전 파일의 '08시' 열 이름을 '~08시'로 맞춰서 읽는다
manifest = pd.read_csv(ROOT / 'logs' / 'topis_manifest.csv', encoding='utf-8-sig')
print(manifest['상태'].value_counts().to_dict())
manifest[['연월', '상태', '행수', '링크수', '날짜수', '결측률', '메모']]

{'OK': 41, 'WARN': 4}


,연월,상태,행수,링크수,날짜수,결측률,메모
0,2022-08,OK,156677,5059,31,0.0008,NaN
1,2022-11,OK,151758,5062,30,0.0009,NaN
2,2022-12,OK,156736,5056,31,0.0007,NaN
3,2023-01,OK,156814,5062,31,0.0004,NaN
4,2023-02,OK,141736,5062,28,0.0003,NaN
5,2023-03,OK,156938,5063,31,0.0003,NaN
6,2023-04,OK,151854,5063,30,0.0005,NaN
7,2023-05,OK,156891,5061,31,0.0004,NaN
8,2023-06,OK,151823,5061,30,0.0003,NaN
9,2023-07,OK,156876,5061,31,0.0004,NaN


## 1. `~08시` 열은 몇 시의 평균인가

엑셀에는 `~08시`가 07~08시인지 08~09시인지 나와 있지 않다.
두 해석으로 각각 평균을 내서 TOPIS 월간 보고서(4쪽 요일별 표의 평일 값)와 비교한다.
열 이름이 바뀐 전후를 모두 보려고 2024년 1월(`08시` 형식)과 2026년 1월(`~08시` 형식)을 확인한다.
보고서가 어떤 방식으로 평균을 냈는지 모르므로 단순 평균과 총거리/총시간을 둘 다 계산한다.

In [2]:
periods = {'전일 06-22': (6, 22), '오전 07-09': (7, 9), '낮 12-14': (12, 14), '오후 17-19': (17, 19)}
reports = {  # 각 달 보고서의 평일 값
    '2024-01': ([22.6, 23.4, 23.1, 19.7], 20240101),
    '2026-01': ([22.0, 22.7, 22.6, 19.0], 20260101),
}


def period_speed(df, start, end, shift):
    # shift=1 이면 '~HH시' = (HH-1)~HH시, shift=0 이면 HH~(HH+1)시
    cols = [f'~{h + shift:02d}시' for h in range(start, end)]
    v = df[cols].stack().dropna()
    s = v.to_numpy(dtype=float)
    d = df.loc[v.index.get_level_values(0), '거리'].to_numpy()
    return s.mean(), d.sum() / (d / s).sum()


rows = []
for ym, (values, holiday) in reports.items():
    df = read_topis(ym)
    df = df[df['요일'].isin(['월', '화', '수', '목', '금']) & (df['일자'] != holiday)]
    for name, shift in [('A: 07~08시', 1), ('B: 08~09시', 0)]:
        for (p, (a, b)), rep in zip(periods.items(), values):
            simple, space = period_speed(df, a, b, shift)
            rows.append({'연월': ym, '해석(~08시)': name, '시간대': p, '보고서': rep,
                         '단순평균': round(simple, 2), '총거리/총시간': round(space, 2)})
check = pd.DataFrame(rows)
check['차이(총거리/총시간)'] = (check['총거리/총시간'] - check['보고서']).abs().round(2)
check

,연월,해석(~08시),시간대,보고서,단순평균,총거리/총시간,차이(총거리/총시간)
0,2024-01,A: 07~08시,전일 06-22,22.6,24.51,22.44,0.16
1,2024-01,A: 07~08시,오전 07-09,23.4,25.06,23.30,0.10
2,2024-01,A: 07~08시,낮 12-14,23.1,24.93,23.06,0.04
3,2024-01,A: 07~08시,오후 17-19,19.7,21.93,19.64,0.06
4,2024-01,B: 08~09시,전일 06-22,22.6,24.76,22.61,0.01
5,2024-01,B: 08~09시,오전 07-09,23.4,27.18,25.61,2.21
6,2024-01,B: 08~09시,낮 12-14,23.1,24.82,22.88,0.22
7,2024-01,B: 08~09시,오후 17-19,19.7,22.78,20.68,0.98
8,2026-01,A: 07~08시,전일 06-22,22.0,24.01,21.85,0.15
9,2026-01,A: 07~08시,오전 07-09,22.7,24.52,22.65,0.05


In [3]:
check.groupby(['연월', '해석(~08시)'])['차이(총거리/총시간)'].mean().round(2)

연월       해석(~08시) 
2024-01  A: 07~08시    0.09
         B: 08~09시    0.86
2026-01  A: 07~08시    0.07
         B: 08~09시    0.88
Name: 차이(총거리/총시간), dtype: float64

결과: 두 달 모두 해석 A(`~08시` = 07~08시)에 총거리/총시간으로 계산하면 보고서와 0.1km/h 안쪽으로 맞는다.
2024-02 이전 파일의 `08시`도 뜻은 같고 이름만 다르다.

- 출근 07~09시 -> `~08시`, `~09시`
- 퇴근 17~20시 -> `~18시`, `~19시`, `~20시`

보고서도 속도를 총거리/총시간으로 계산하므로, 통행시간은 구간별 (거리 / 속도)를 더하는 방식이 맞다.

## 2. 서울 전체가 동시에 비는 시간

결측률이 1%를 넘은 달을 날짜별로 보면 특정 이틀에 몰려 있었다.
45개월 전체에서 한 시간 동안 전체 링크의 90% 이상이 비어 있는 시간을 찾는다.

In [4]:
gaps = []
for f in sorted(TOPIS_CSV_DIR.glob('topis_speed_*.csv')):
    df = read_topis(f)[['일자'] + HOUR_COLS]
    ratio = df[HOUR_COLS].isna().groupby(df['일자']).mean()
    hit = ratio.stack()
    hit = hit[hit >= 0.9]
    for (day, col), r in hit.items():
        h = int(col[1:3])
        gaps.append({'date': pd.to_datetime(str(day)).strftime('%Y-%m-%d'), 'hour': h,
                     'start': f'{h - 1:02d}:00', 'end': f'{h:02d}:00',
                     'missing_ratio': round(r, 3)})

gaps = pd.DataFrame(gaps)
gaps['weekday'] = pd.to_datetime(gaps['date']).dt.day_name()
print(len(gaps), '시간')
gaps.groupby(['date', 'weekday']).agg(시간수=('hour', 'size'), 처음=('start', 'min'), 끝=('end', 'max'),
                                     최소결측=('missing_ratio', 'min'))

41 시간


,,시간수,처음,끝,최소결측
date,weekday,,,,
2024-08-02,Friday,3,21:00,24:00,1.0
2024-08-03,Saturday,7,00:00,07:00,1.0
2024-11-29,Friday,3,21:00,24:00,1.0
2024-11-30,Saturday,5,00:00,05:00,1.0
2025-07-04,Friday,4,20:00,24:00,1.0
2025-07-05,Saturday,8,00:00,08:00,1.0
2026-07-03,Friday,4,20:00,24:00,1.0
2026-07-04,Saturday,7,00:00,07:00,1.0


공백은 모두 금요일 밤~토요일 아침이고 도시고속도로(검지기)와 일반도로(택시 GPS)가 함께 비어 있다.
교통 상황보다는 수집 시스템 공백으로 추정된다. 분석에서 뺄지는 팀에서 정한다.

`hour`는 TOPIS 열의 끝 시각이다 (`config/periods.yaml`의 commute_hours와 같은 기준). 예: hour=8 -> `~08시` 열 = 07:00~08:00

`datetime`도 끝 시각으로 적는다. 날씨 표(weather_hourly)와 datetime으로 바로 붙일 수 있다. `~24시`는 다음 날 00:00이다.

팀 결정(11/20): 평소값 · 통행시간 계산에서 이 시간은 행을 지우지 않고 NaN으로 둔다.

In [5]:
gaps['datetime'] = pd.to_datetime(gaps['date']) + pd.to_timedelta(gaps['hour'], unit='h')
gaps['reason'] = '서울 전체 링크 동시 결측 (수집 시스템 공백 추정)'
out = gaps[['date', 'hour', 'datetime', 'start', 'end', 'missing_ratio', 'reason']]
out.to_csv(ROOT / 'config' / 'topis_gaps.csv', index=False, encoding='utf-8-sig')
out.head(10)

,date,hour,datetime,start,end,missing_ratio,reason
0,2024-08-02,22,2024-08-02 22:00:00,21:00,22:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
1,2024-08-02,23,2024-08-02 23:00:00,22:00,23:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
2,2024-08-02,24,2024-08-03 00:00:00,23:00,24:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
3,2024-08-03,1,2024-08-03 01:00:00,00:00,01:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
4,2024-08-03,2,2024-08-03 02:00:00,01:00,02:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
5,2024-08-03,3,2024-08-03 03:00:00,02:00,03:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
6,2024-08-03,4,2024-08-03 04:00:00,03:00,04:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
7,2024-08-03,5,2024-08-03 05:00:00,04:00,05:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
8,2024-08-03,6,2024-08-03 06:00:00,05:00,06:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
9,2024-08-03,7,2024-08-03 07:00:00,06:00,07:00,1.0,서울 전체 링크 동시 결측 (수집 시스템 공백 추정)
